In [ ]:
# Cell 1 — Config

In [1]:
import os
import math
import json
import time
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import GPT2Tokenizer
from functools import partial

torch.manual_seed(42)
random.seed(42)
torch.backends.cudnn.benchmark = True   # NEW: speeds up fixed-shape conv/attention kernels

BASE_DIR = r"C:\Users\MIND & MATTER\Desktop\ScratchLLM-95_conversational"
DATA_DIR = os.path.join(BASE_DIR, "data", "processed", "New folder")

# FIXED: point to the actual train/val split files, not the full combined file
TRAIN_PATH = os.path.join(DATA_DIR, "chatalpaca_train_converted.jsonl")
VAL_PATH   = os.path.join(DATA_DIR, "chatalpaca_val_converted.jsonl")

MERGED_BASE_PATH = os.path.join(BASE_DIR, "checkpoints", "stage_a_merged", "merged.pt")

CKPT_DIR = os.path.join(BASE_DIR, "checkpoints", "stage_b_v2")
os.makedirs(os.path.join(CKPT_DIR, "best"), exist_ok=True)
LATEST_CKPT = os.path.join(CKPT_DIR, "latest.pt")
BEST_CKPT = os.path.join(CKPT_DIR, "best", "best.pt")

CONFIG = {
    "vocab_size": 50257, "max_seq_len": 512, "embed_dim": 512,
    "num_heads": 8, "num_layers": 12, "d_ff": 2730, "dropout": 0.1,

    "lora_r": 64, "lora_alpha": 128, "lora_dropout": 0.05,
    "target_modules": ["qkv_proj", "out_proj", "w1", "w2", "w3"],  # attention + FFN

    "batch_size": 8, "grad_accum_steps": 4,   # NEW: same effective batch (32), less loop overhead
    "lr": 2e-4, "warmup_steps": 300, "epochs": 5,
    "grad_clip": 0.8, "eval_every_steps": 200, "patience": 3,

    "device": "cuda" if torch.cuda.is_available() else "cpu",
}

print("Device:", CONFIG["device"])
print(CONFIG)

Device: cuda
{'vocab_size': 50257, 'max_seq_len': 512, 'embed_dim': 512, 'num_heads': 8, 'num_layers': 12, 'd_ff': 2730, 'dropout': 0.1, 'lora_r': 64, 'lora_alpha': 128, 'lora_dropout': 0.05, 'target_modules': ['qkv_proj', 'out_proj', 'w1', 'w2', 'w3'], 'batch_size': 8, 'grad_accum_steps': 4, 'lr': 0.0002, 'warmup_steps': 300, 'epochs': 5, 'grad_clip': 0.8, 'eval_every_steps': 200, 'patience': 3, 'device': 'cuda'}


In [2]:
# Cell 2 — Model architecture

In [3]:
class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_seq_len):
        super().__init__()
        inv_freq = 1.0 / (10000 ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq)
        t = torch.arange(max_seq_len).float()
        freqs = torch.einsum("i,j->ij", t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer("cos_cached", emb.cos()[None, None, :, :])
        self.register_buffer("sin_cached", emb.sin()[None, None, :, :])

    def forward(self, x, seq_len):
        return self.cos_cached[:, :, :seq_len, :], self.sin_cached[:, :, :seq_len, :]

def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)

def apply_rope(q, k, cos, sin):
    q = (q * cos) + (rotate_half(q) * sin)
    k = (k * cos) + (rotate_half(k) * sin)
    return q, k

class SwiGLU(nn.Module):
    def __init__(self, dim, d_ff, dropout):
        super().__init__()
        self.w1 = nn.Linear(dim, d_ff, bias=False)
        self.w2 = nn.Linear(dim, d_ff, bias=False)
        self.w3 = nn.Linear(d_ff, dim, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.w3(F.silu(self.w1(x)) * self.w2(x)))

class CausalSelfAttention(nn.Module):
    def __init__(self, dim, num_heads, max_seq_len, dropout):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.qkv_proj = nn.Linear(dim, dim * 3, bias=False)
        self.out_proj = nn.Linear(dim, dim, bias=False)
        self.rope = RotaryEmbedding(self.head_dim, max_seq_len)
        self.dropout = dropout

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv_proj(x).reshape(B, T, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        cos, sin = self.rope(x, T)
        q, k = apply_rope(q, k, cos, sin)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=self.dropout if self.training else 0.0)
        out = out.transpose(1, 2).reshape(B, T, C)
        return self.out_proj(out)

class TransformerBlock(nn.Module):
    def __init__(self, dim, num_heads, d_ff, max_seq_len, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(dim)
        self.attn = CausalSelfAttention(dim, num_heads, max_seq_len, dropout)
        self.ln2 = nn.LayerNorm(dim)
        self.ffn = SwiGLU(dim, d_ff, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x

class ENGLlmModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.token_embed = nn.Embedding(cfg["vocab_size"], cfg["embed_dim"])
        self.blocks = nn.ModuleList([
            TransformerBlock(cfg["embed_dim"], cfg["num_heads"], cfg["d_ff"], cfg["max_seq_len"], cfg["dropout"])
            for _ in range(cfg["num_layers"])
        ])
        self.ln_f = nn.LayerNorm(cfg["embed_dim"])
        self.lm_head = nn.Linear(cfg["embed_dim"], cfg["vocab_size"], bias=False)

    def forward(self, input_ids):
        x = self.token_embed(input_ids)
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        return self.lm_head(x)

print("Model architecture defined.")

Model architecture defined.


In [4]:
# Cell 3 — LoRA wrapper (attention + FFN)

In [5]:
class LoRALinear(nn.Module):
    def __init__(self, base_linear, r, alpha, dropout):
        super().__init__()
        self.base = base_linear
        self.base.weight.requires_grad = False
        in_dim = base_linear.in_features
        out_dim = base_linear.out_features
        self.lora_A = nn.Parameter(torch.zeros(r, in_dim))
        self.lora_B = nn.Parameter(torch.zeros(out_dim, r))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        self.scaling = alpha / r
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        base_out = self.base(x)
        lora_out = self.dropout(x) @ self.lora_A.T @ self.lora_B.T
        return base_out + lora_out * self.scaling

def apply_lora(model, target_modules, r, alpha, dropout):
    for block in model.blocks:
        if "qkv_proj" in target_modules:
            block.attn.qkv_proj = LoRALinear(block.attn.qkv_proj, r, alpha, dropout)
        if "out_proj" in target_modules:
            block.attn.out_proj = LoRALinear(block.attn.out_proj, r, alpha, dropout)
        if "w1" in target_modules:
            block.ffn.w1 = LoRALinear(block.ffn.w1, r, alpha, dropout)
        if "w2" in target_modules:
            block.ffn.w2 = LoRALinear(block.ffn.w2, r, alpha, dropout)
        if "w3" in target_modules:
            block.ffn.w3 = LoRALinear(block.ffn.w3, r, alpha, dropout)
    return model

print("LoRA wrapper defined (attention + FFN).")

LoRA wrapper defined (attention + FFN).


In [6]:
# Cell 4 — Load Stage A merged checkpoint, apply fresh LoRA

In [7]:
# Add this BEFORE torch.load
class GPTConfig:
    def __init__(self, *args, **kwargs):
        self.__dict__.update(kwargs)
    # add whatever fields/methods the original class had

model = ENGLlmModel(CONFIG)

checkpoint = torch.load(MERGED_BASE_PATH, map_location="cpu", weights_only=False)
state_dict = checkpoint["model_state_dict"] if "model_state_dict" in checkpoint else checkpoint

missing, unexpected = model.load_state_dict(state_dict, strict=False)
print("Missing keys:", missing)
print("Unexpected keys:", unexpected)

for p in model.parameters():
    p.requires_grad = False

model = apply_lora(model, CONFIG["target_modules"], CONFIG["lora_r"], CONFIG["lora_alpha"], CONFIG["lora_dropout"])
model.to(CONFIG["device"])

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} ({100*trainable/total:.2f}% of {total:,})")

Missing keys: []
Unexpected keys: []
Trainable params: 9,828,864 (7.91% of 124,219,904)


In [8]:
# Cell 5 — Dataset & collate_fn (correctly shifted labels)

In [9]:
import json, random

SRC = r"C:\Users\MIND & MATTER\Desktop\ScratchLLM-95_conversational\data\processed\New folder\chatalpaca_converted.jsonl"
TRAIN_OUT = SRC.replace("chatalpaca_converted", "chatalpaca_train_converted")
VAL_OUT = SRC.replace("chatalpaca_converted", "chatalpaca_val_converted")

with open(SRC, encoding="utf-8") as f:
    lines = [l for l in f if l.strip()]

random.seed(42)
random.shuffle(lines)
val_frac = 0.02  # 2% val split, adjust as needed
n_val = max(1, int(len(lines) * val_frac))

val_lines = lines[:n_val]
train_lines = lines[n_val:]

with open(TRAIN_OUT, "w", encoding="utf-8") as f:
    f.writelines(train_lines)
with open(VAL_OUT, "w", encoding="utf-8") as f:
    f.writelines(val_lines)

print(f"Train: {len(train_lines)} | Val: {len(val_lines)}")

Train: 67836 | Val: 1384


In [10]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

class JsonlTextDataset(Dataset):
    def __init__(self, path, tokenizer, max_len):
        self.examples = []
        with open(path, encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                text = json.loads(line)["text"] + tokenizer.eos_token
                ids = tokenizer.encode(text, truncation=True, max_length=max_len)
                self.examples.append(ids)

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]

def collate_fn(batch, pad_id, max_len):
    max_batch_len = min(max(len(x) for x in batch), max_len)
    input_ids, labels = [], []
    for ids in batch:
        ids = ids[:max_batch_len]
        pad_len = max_batch_len - len(ids)
        input_ids.append(ids + [pad_id] * pad_len)
        shifted = ids[1:] + [-100] * (pad_len + 1)
        labels.append(shifted[:max_batch_len])
    return torch.tensor(input_ids), torch.tensor(labels)

train_dataset = JsonlTextDataset(TRAIN_PATH, tokenizer, CONFIG["max_seq_len"])
val_dataset = JsonlTextDataset(VAL_PATH, tokenizer, CONFIG["max_seq_len"])

print(f"Train examples: {len(train_dataset)}")
print(f"Val examples: {len(val_dataset)}")

collate = partial(collate_fn, pad_id=tokenizer.eos_token_id, max_len=CONFIG["max_seq_len"])

# NEW: num_workers + pin_memory + persistent_workers for faster data feeding
train_loader = DataLoader(
    train_dataset, batch_size=CONFIG["batch_size"], shuffle=True,
    collate_fn=collate, num_workers=0, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=CONFIG["batch_size"], shuffle=False,
    collate_fn=collate, num_workers=0, pin_memory=True
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

Train examples: 67836
Val examples: 1384
Train batches: 8480
Val batches: 173


In [11]:
# Cell 6 — Sanity check (run before training, every time)

In [12]:
batch = next(iter(train_loader))
input_ids, labels = batch
print("input_ids[0][:10]:", input_ids[0][:10].tolist())
print("labels[0][:10]:   ", labels[0][:10].tolist())
print("Match:", input_ids[0][1:10].tolist() == labels[0][:9].tolist())

input_ids[0][:10]: [21017, 46486, 25, 198, 15056, 262, 4634, 11, 2251, 257]
labels[0][:10]:    [46486, 25, 198, 15056, 262, 4634, 11, 2251, 257, 36102]
Match: True


In [13]:
# Cell 7 — Optimizer, scheduler, loss

In [14]:
trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=CONFIG["lr"], betas=(0.9, 0.95), weight_decay=0.01)

total_steps = (len(train_loader) // CONFIG["grad_accum_steps"]) * CONFIG["epochs"]

def lr_lambda(step):
    if step < CONFIG["warmup_steps"]:
        return step / max(1, CONFIG["warmup_steps"])
    progress = (step - CONFIG["warmup_steps"]) / max(1, total_steps - CONFIG["warmup_steps"])
    return 0.5 * (1 + math.cos(math.pi * min(progress, 1.0)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

def compute_loss(logits, labels):
    return F.cross_entropy(logits.view(-1, logits.size(-1)), labels.view(-1), ignore_index=-100)

print(f"Total optimizer steps planned: {total_steps}")

Total optimizer steps planned: 10600


In [15]:
# Cell 8 — Checkpoint helpers

In [16]:
def save_checkpoint(path, step, epoch, best_val_loss):
    torch.save({
        "step": step, "epoch": epoch, "best_val_loss": best_val_loss,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
    }, path)

def load_checkpoint_if_exists():
    if not os.path.exists(LATEST_CKPT):
        print("No existing checkpoint found, starting fresh.")
        return 0, 0, float("inf")
    ckpt = torch.load(LATEST_CKPT, map_location=CONFIG["device"], weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    print(f"Resumed from step {ckpt['step']}, epoch {ckpt['epoch']}, best_val_loss {ckpt['best_val_loss']:.4f}")
    return ckpt["step"], ckpt["epoch"], ckpt["best_val_loss"]

In [17]:
# Cell 9 — Eval function

In [18]:
@torch.no_grad()
def evaluate():
    model.eval()
    total_loss, total_batches = 0.0, 0
    use_amp = CONFIG["device"] == "cuda"
    amp_dtype = torch.bfloat16 if (use_amp and torch.cuda.is_bf16_supported()) else torch.float16

    for input_ids, labels in val_loader:
        input_ids, labels = input_ids.to(CONFIG["device"]), labels.to(CONFIG["device"])
        with torch.autocast(device_type="cuda", dtype=amp_dtype, enabled=use_amp):
            logits = model(input_ids)
            loss = compute_loss(logits, labels)
        total_loss += loss.item()
        total_batches += 1

    model.train()
    avg_loss = total_loss / total_batches
    return avg_loss, math.exp(avg_loss)

In [19]:
# Cell 10 — Sample generation function

In [20]:
@torch.no_grad()
def generate_sample(prompt, max_new_tokens=60, repetition_penalty=1.3, window=128):
    model.eval()
    ids = tokenizer.encode(prompt)
    input_ids = torch.tensor([ids]).to(CONFIG["device"])
    generated = list(ids)
    for _ in range(max_new_tokens):
        input_ids_trunc = input_ids[:, -CONFIG["max_seq_len"]:]
        logits = model(input_ids_trunc)[:, -1, :]
        recent = generated[-window:]
        for tok_id in set(recent):
            logits[0, tok_id] /= repetition_penalty
        next_id = torch.argmax(logits, dim=-1, keepdim=True)
        input_ids = torch.cat([input_ids, next_id], dim=1)
        generated.append(next_id.item())
        if next_id.item() == tokenizer.eos_token_id:
            break
    model.train()
    return tokenizer.decode(input_ids[0], skip_special_tokens=True)

In [ ]:
# Cell 11 — RESUME cell (run every time, before training)

In [22]:
start_step, start_epoch, best_val_loss = load_checkpoint_if_exists()
global_step = start_step
print(f"Will resume from global_step={global_step}, epoch={start_epoch}")

Resumed from step 10600, epoch 4, best_val_loss 3.2513
Will resume from global_step=10600, epoch=4


In [ ]:
# Cell 12 — Training loop

In [ ]:
model.train()
patience_counter = 0
SAMPLE_PROMPT = "### Instruction:\nWhat should I do this weekend?\n\n### Response:\n"

use_amp = CONFIG["device"] == "cuda"
amp_dtype = torch.bfloat16 if (use_amp and torch.cuda.is_bf16_supported()) else torch.float16
scaler = torch.cuda.amp.GradScaler(enabled=(use_amp and amp_dtype == torch.float16))

for epoch in range(start_epoch, CONFIG["epochs"]):
    epoch_start_time = time.time()
    optimizer.zero_grad()

    for i, (input_ids, labels) in enumerate(train_loader):
        input_ids = input_ids.to(CONFIG["device"], non_blocking=True)
        labels = labels.to(CONFIG["device"], non_blocking=True)

        with torch.autocast(device_type="cuda", dtype=amp_dtype, enabled=use_amp):
            logits = model(input_ids)
            loss = compute_loss(logits, labels) / CONFIG["grad_accum_steps"]

        scaler.scale(loss).backward()

        if (i + 1) % CONFIG["grad_accum_steps"] == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable_params, CONFIG["grad_clip"])
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()
            global_step += 1

            if global_step % 50 == 0:
                step_loss = loss.item() * CONFIG["grad_accum_steps"]
                print(f"Epoch {epoch} | Step {global_step} | loss {step_loss:.4f} | ppl {math.exp(step_loss):.2f} | lr {scheduler.get_last_lr()[0]:.2e}")

            if global_step % CONFIG["eval_every_steps"] == 0:
                val_loss, val_ppl = evaluate()
                print(f"  [EVAL] Step {global_step} | val_loss {val_loss:.4f} | val_ppl {val_ppl:.2f}")

                is_best = val_loss < best_val_loss
                if is_best:
                    best_val_loss = val_loss
                    patience_counter = 0
                    save_checkpoint(BEST_CKPT, global_step, epoch, best_val_loss)
                    print("  -> new best, saved")
                else:
                    patience_counter += 1
                    print(f"  -> no improvement ({patience_counter}/{CONFIG['patience']})")

                save_checkpoint(LATEST_CKPT, global_step, epoch, best_val_loss)

                sample = generate_sample(SAMPLE_PROMPT)
                print(f"\n  [SAMPLE @ step {global_step}]\n{sample}\n")

                if patience_counter >= CONFIG["patience"]:
                    print("Early stopping triggered.")
                    break
    else:
        epoch_time = time.time() - epoch_start_time
        print(f"\n=== Epoch {epoch} complete in {epoch_time/60:.1f} min ===")
        save_checkpoint(LATEST_CKPT, global_step, epoch + 1, best_val_loss)
        continue
    break

print("\nTraining finished (completed or early-stopped). Best checkpoint saved at:", BEST_CKPT)

In [ ]:
# --- GPU info + timing check (run this ONCE, not the full training loop yet) ---
import time, torch

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU found")
if torch.cuda.is_available():
    print("Total VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

N_STEPS_TO_TIME = 20   # number of optimizer steps to time (not batches)

use_amp = CONFIG["device"] == "cuda"
amp_dtype = torch.bfloat16 if (use_amp and torch.cuda.is_bf16_supported()) else torch.float16
_scaler = torch.cuda.amp.GradScaler(enabled=(use_amp and amp_dtype == torch.float16))

model.train()
optimizer.zero_grad()

if torch.cuda.is_available():
    torch.cuda.synchronize()
t0 = time.time()

steps_done = 0
for i, (input_ids, labels) in enumerate(train_loader):
    input_ids = input_ids.to(CONFIG["device"], non_blocking=True)
    labels = labels.to(CONFIG["device"], non_blocking=True)

    with torch.autocast(device_type="cuda", dtype=amp_dtype, enabled=use_amp):
        logits = model(input_ids)
        loss = compute_loss(logits, labels) / CONFIG["grad_accum_steps"]

    _scaler.scale(loss).backward()

    if (i + 1) % CONFIG["grad_accum_steps"] == 0:
        _scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(trainable_params, CONFIG["grad_clip"])
        _scaler.step(optimizer)
        _scaler.update()
        optimizer.zero_grad()
        steps_done += 1

    if steps_done >= N_STEPS_TO_TIME:
        break

if torch.cuda.is_available():
    torch.cuda.synchronize()
t1 = time.time()

elapsed = t1 - t0
print(f"\nTimed {steps_done} optimizer steps in {elapsed:.2f}s")
print(f"Time per optimizer step: {elapsed/steps_done:.3f}s")
print(f"Estimated time for 200 steps: {(elapsed/steps_done)*200/60:.2f} min")
print(f"Estimated time for one full epoch ({total_steps//CONFIG['epochs']} steps): {(elapsed/steps_done)*(total_steps//CONFIG['epochs'])/60:.1f} min")

In [ ]:
# Reload the ORIGINAL best checkpoint (undo the EOS fine-tune attempt)
ckpt = torch.load(BEST_CKPT, map_location=CONFIG["device"], weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.to(CONFIG["device"])
print(f"Reloaded original best checkpoint from step {ckpt['step']}, val_loss {ckpt['best_val_loss']:.4f}")

In [ ]:
@torch.no_grad()
def generate_sample(prompt, max_new_tokens=120, repetition_penalty=1.3, window=128, min_len=40):
    model.eval()
    ids = tokenizer.encode(prompt)
    input_ids = torch.tensor([ids]).to(CONFIG["device"])
    generated = list(ids)
    prompt_len = len(ids)

    for _ in range(max_new_tokens):
        input_ids_trunc = input_ids[:, -CONFIG["max_seq_len"]:]
        logits = model(input_ids_trunc)[:, -1, :]
        recent = generated[-window:]
        for tok_id in set(recent):
            logits[0, tok_id] /= repetition_penalty
        next_id = torch.argmax(logits, dim=-1, keepdim=True)
        input_ids = torch.cat([input_ids, next_id], dim=1)
        generated.append(next_id.item())

        decoded_new = tokenizer.decode(generated[prompt_len:])
        sentence_count = decoded_new.count(".") + decoded_new.count("!") + decoded_new.count("?")

        if next_id.item() == tokenizer.eos_token_id:
            break
        # require enough length AND at least 2 sentences before allowing punctuation-based stop
        if len(generated) - prompt_len > min_len and sentence_count >= 2 and decoded_new.rstrip().endswith((".", "!", "?")):
            break
        if "### Instruction" in decoded_new[-30:]:
            break

    model.train()
    return tokenizer.decode(input_ids[0], skip_special_tokens=True)

In [ ]:
test_prompts = [
    "### Instruction:\nWhat should I do this weekend?\n\n### Response:\n",
    "### Instruction:\nExplain how photosynthesis works.\n\n### Response:\n",
    "### Instruction:\nWrite a short poem about the ocean.\n\n### Response:\n",
]
for p in test_prompts:
    print(generate_sample(p))
    print("---")

In [ ]:
test_prompts = [
    "### Instruction:\nWhat should I do this weekend?\n\n### Response:\n",
    "### Instruction:\nExplain how photosynthesis works.\n\n### Response:\n",
    "### Instruction:\nWrite a short poem about the ocean.\n\n### Response:\n",
]
for p in test_prompts:
    print(generate_sample(p))
    print("---")

In [ ]:
import copy

@torch.no_grad()
def merge_lora_weights(model):
    for block in model.blocks:
        for name in ["qkv_proj", "out_proj"]:
            layer = getattr(block.attn, name)
            if hasattr(layer, "lora_A"):
                merged_weight = layer.base.weight + (layer.lora_B @ layer.lora_A) * layer.scaling
                new_linear = nn.Linear(layer.base.in_features, layer.base.out_features, bias=False)
                new_linear.weight.copy_(merged_weight)
                setattr(block.attn, name, new_linear)
        for name in ["w1", "w2", "w3"]:
            layer = getattr(block.ffn, name)
            if hasattr(layer, "lora_A"):
                merged_weight = layer.base.weight + (layer.lora_B @ layer.lora_A) * layer.scaling
                new_linear = nn.Linear(layer.base.in_features, layer.base.out_features, bias=False)
                new_linear.weight.copy_(merged_weight)
                setattr(block.ffn, name, new_linear)
    return model

# Work on a copy so your training-session `model` object stays untouched
merged_model = copy.deepcopy(model)
merged_model = merge_lora_weights(merged_model)
merged_model.eval()
print("LoRA weights merged into base model.")

In [ ]:
FINAL_MODEL_PATH = os.path.join(BASE_DIR, "checkpoints", "final_conversational_model.pt")
torch.save({
    "model_state_dict": merged_model.state_dict(),
    "config": CONFIG,
}, FINAL_MODEL_PATH)
print(f"Final merged model saved to: {FINAL_MODEL_PATH}")

In [ ]:
merged_model = copy.deepcopy(model)
merged_model = merge_lora_weights(merged_model)
merged_model = merged_model.to(CONFIG["device"])   # <-- add this line
merged_model.eval()
print("LoRA weights merged into base model.")

In [ ]:
@torch.no_grad()
def chat(prompt_text, max_new_tokens=120, repetition_penalty=1.3, min_len=40):
    merged_model.eval()
    full_prompt = f"### Instruction:\n{prompt_text}\n\n### Response:\n"
    ids = tokenizer.encode(full_prompt)
    input_ids = torch.tensor([ids]).to(CONFIG["device"])
    generated = list(ids)
    prompt_len = len(ids)

    for _ in range(max_new_tokens):
        input_ids_trunc = input_ids[:, -CONFIG["max_seq_len"]:]
        logits = merged_model(input_ids_trunc)[:, -1, :]
        recent = generated[-128:]
        for tok_id in set(recent):
            logits[0, tok_id] /= repetition_penalty
        next_id = torch.argmax(logits, dim=-1, keepdim=True)
        input_ids = torch.cat([input_ids, next_id], dim=1)
        generated.append(next_id.item())

        decoded_new = tokenizer.decode(generated[prompt_len:])
        sentence_count = decoded_new.count(".") + decoded_new.count("!") + decoded_new.count("?")

        if next_id.item() == tokenizer.eos_token_id:
            break
        if len(generated) - prompt_len > min_len and sentence_count >= 2 and decoded_new.rstrip().endswith((".", "!", "?")):
            break
        if "### Instruction" in decoded_new[-30:]:
            break

    return tokenizer.decode(generated[prompt_len:], skip_special_tokens=True).strip()

# Test it
print(chat("What should I do this weekend?"))

In [ ]:
test_prompt = "### Instruction:\nWhat should I do this weekend?\n\n### Response:\n"
ids = tokenizer.encode(test_prompt)
input_ids = torch.tensor([ids]).to(CONFIG["device"])

model.eval()
with torch.no_grad():
    logits_original = model(input_ids)[:, -1, :]
    logits_merged = merged_model(input_ids)[:, -1, :]

diff = (logits_original - logits_merged).abs().max().item()
print(f"Max logit difference between original (LoRA) and merged model: {diff:.6f}")
# Should be extremely small (e.g. < 1e-4), confirming merge is mathematically correct
model.train()

In [ ]:
model.state_dict()  # convert to safetensors instead of raw pt for HF
from safetensors.torch import save_file
save_file(merged_model.state_dict(), "model.safetensors")

In [ ]:
import os
print(os.getcwd())

In [23]:
import os
from safetensors.torch import save_file

FINAL_SAFETENSORS_PATH = r"C:\Users\MIND & MATTER\Desktop\ScratchLLM-95_conversational\checkpoints\model.safetensors"

sd = {k: v.contiguous() for k, v in merged_model.state_dict().items()}
save_file(sd, FINAL_SAFETENSORS_PATH)
print(f"Saved to: {FINAL_SAFETENSORS_PATH}")

NameError: name 'merged_model' is not defined

In [ ]:
FULL_CKPT_PATH = os.path.join(BASE_DIR, "checkpoints", "final_full_checkpoint.pt")

torch.save({
    "model_state_dict": merged_model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "scheduler_state_dict": scheduler.state_dict(),
    "config": CONFIG,
    "step": global_step,
    "best_val_loss": best_val_loss,
}, FULL_CKPT_PATH)

size_mb = os.path.getsize(FULL_CKPT_PATH) / (1024 * 1024)
print(f"Full checkpoint saved: {FULL_CKPT_PATH}")
print(f"Size: {size_mb:.1f} MB")

In [ ]:
from safetensors.torch import save_file

SAFETENSORS_PATH = os.path.join(BASE_DIR, "checkpoints", "model.safetensors")
sd = {k: v.contiguous() for k, v in merged_model.state_dict().items()}
save_file(sd, SAFETENSORS_PATH)
print(f"Weights-only safetensors saved: {SAFETENSORS_PATH}")
print(f"Size: {os.path.getsize(SAFETENSORS_PATH)/(1024*1024):.1f} MB")

In [ ]:
import shutil

HF_UPLOAD_DIR = os.path.join(BASE_DIR, "hf_upload")
os.makedirs(HF_UPLOAD_DIR, exist_ok=True)

shutil.copy(SAFETENSORS_PATH, os.path.join(HF_UPLOAD_DIR, "model.safetensors"))
shutil.copy(FULL_CKPT_PATH, os.path.join(HF_UPLOAD_DIR, "full_checkpoint.pt"))

print("Files ready for upload in:", HF_UPLOAD_DIR)
print(os.listdir(HF_UPLOAD_DIR))

In [ ]:
readme_note = """
## Files in this repo

- `model.safetensors` — inference-ready model weights only (recommended for loading/using the model)
- `full_checkpoint.pt` — complete training checkpoint including optimizer & scheduler state (for resuming training or archival purposes; NOT needed for inference)
"""
with open(os.path.join(HF_UPLOAD_DIR, "FILES.md"), "w") as f:
    f.write(readme_note)
print("FILES.md written.")